In [1]:
%load_ext autoreload
%autoreload 2

# RAG exploration

Tests `backend/retriever.py`, `backend/generator.py`, and `backend/rag.py` against the real ChromaDB index built by the ingestion notebook, and a real HF token.(though limit of it is reached.)

In [2]:
import sys
sys.path.append("..")

from backend import retriever, generator, rag
from pipeline import config

assert config.HF_TOKEN, "no HF_TOKEN found, check your .env file"

d:\Aanaconda\envs\ukask\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1, retriever alone

Check what actually comes back for a question before involving the LLM at all, this is the easiest place to catch a bad embedding or a chunking problem.

In [3]:
query = "شرایط حذف و اضافه چیست؟"
results = retriever.search(query, k=5)

for r in results:
    m = r["metadata"]
    print(f"dist={r['distance']:.3f}  page={m['page']}  article={m['article_numbers']}  type={m['chunk_type']}")
    print(r["text"])
    print("---")

    loading embedding model, ~2GB, first run downloads it, be patient...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 9559.01it/s]


    embedding model ready
dist=0.429  page=62  article=unknown  type=table
|یشزومآ یاهه مان|نیئآ و تاررقم ه|عومجم|Col 4 |Col 5 |Col 6 |Col 7 |Col 8 |<br> 58 |
|---|---|---|---|---|---|---|---|---|
||<br>|<br>|||||||
|حدهای اخذ شده|ی<br>که تعد اد وا|ند به گون<br>ه <br> ا|جو موافقت|درسی دانش|احدهای|ا حذف و|ر دانشگاه ب|**تبصره 2. ** اگ|
||||||<br>||||
|ی دانشجو محاسبه|تحصیلی برا|یک نیمسال|مسال به عنوان|شود، آن نی| 12 <br> واحد|کمتر از|ک نیمسال|دانشجو در ی|
||||||||||
|||. خواهد شد|نشجو لحاظ|طی برای دا|ال مشرو|یک نیمس|مشروطی، |و در صورت|
||||||||<br>||
|از شروع آزمون|تا دو هفت ه قبل|رت اضطرار|ن ماده در صو|تبصره 1 ای|یت مفاد|اند با رعا|نشجو می<br> تو|**تبصره 3. **دا|
|||||<br> <br>|||||
|تعداد واحدهای|شرط آن که|ذف کند، به|ه آموزشی ح|<br>تأیید<br> گرو|ری را با|درس ن|ل، تنها یک|پایانی نیمسا|
|||||||<br>||<br>|
||||||. د نشود|از 12 <br>واح|شجو کمتر|باقی<br> مانده دان|
|<br> <br>|||||||||
|و و<br>تأیید<br> شورای|واست دانشج|یلی با درخ|نیمسال تحص|دروس یک|ف تمام|ص، حذ|شرایط خا|**تبصره 4. **<br> د

## Step 2, generator alone

Reuse the chunks from step 1, so we're only testing the LLM call and prompt, not retrieval again.

In [5]:
answer = generator.generate_answer(query, results)
print(answer)

HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a55ffc6-302dcf707d28dff616b21942;e720ad3b-ef1a-4d14-8b6a-35597a761530)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.

## Step 3, full pipeline

In [20]:
result = rag.answer_question(query, k=5)
print("ANSWER:", result["answer"])
print()
for s in result["sources"]:
    print(f"  similarity={s['similarity']:.3f}  page={s['page']}  article={s['article_numbers']}")

HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a553375-004ac30d44ecfa0228ff0a1d;4e06f6c0-4d94-4eeb-960c-8db95df96297)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.

## Step 4, run all five sample questions from the brief

The actual test set

In [21]:
sample_questions = [
    "شرایط حذف و اضافه چیست؟",
    "حداکثر تعداد واحد مجاز در هر ترم چقدر است؟",
    "شرایط استفاده از سنوات تحصیلی چیست؟",
    "مراحل دریافت گواهی اشتغال به تحصیل چگونه است؟",
    "دانشجوی مشروط چه محدودیت‌هایی دارد؟",
]

for q in sample_questions:
    result = rag.answer_question(q, k=10)
    print("Q:", q)
    print("A:", result["answer"])
    print("=" * 60)

HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a553392-7299ea9d795c926f7f5239a7;19c1fecd-725d-459d-8cad-f0df1ab26ac7)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.

## Step 5, refusal check

Something clearly outside the regulation documents, confirms the model actually says it can't find an answer instead of guessing.

In [7]:
off_topic = "بهترین رستوران کرمان کدام است؟"
result = rag.answer_question(off_topic, k=5)
print(result["answer"])
print()
print("matches expected refusal:", result["answer"].strip() == generator.NOT_FOUND_MESSAGE)

پاسخ این سؤال در اسناد موجود یافت نشد.

matches expected refusal: True
